In [ ]:
"""
----
학습된 best.pt로 val 셋을 추론하여 최종 성능 지표(mAP, Precision, Recall)와
클래스별 confusion matrix(배경 오탐/미검출, 클래스 간 혼동)를 확인한다.
"""

In [ ]:
import numpy as np
import pandas as pd
from ultralytics import YOLO

DATA_YAML = "./data/dataset_stratified_v4/data.yaml"
MODEL_PATH = "./runs/exp_stratified_v4_customaug/weights/best.pt"

In [ ]:
model = YOLO(MODEL_PATH)
results = model.val(data=DATA_YAML, split="val")

print(f"mAP@50: {results.box.map50:.4f}")
print(f"mAP@50-95: {results.box.map:.4f}")
print(f"Precision: {results.box.mp:.4f}")
print(f"Recall: {results.box.mr:.4f}")

In [ ]:
cm = results.confusion_matrix.matrix
names_list = list(results.names.values()) + ["background"]
bg_idx = len(names_list) - 1

bg_confusions = []
for i in range(len(names_list) - 1):
    if cm[i, bg_idx] > 0:
        bg_confusions.append({"type": "약물→배경(미검출)", "class": names_list[i], "count": int(cm[i, bg_idx])})
    if cm[bg_idx, i] > 0:
        bg_confusions.append({"type": "배경→약물(오탐)", "class": names_list[i], "count": int(cm[bg_idx, i])})
bg_df = pd.DataFrame(bg_confusions).sort_values("count", ascending=False) if bg_confusions else pd.DataFrame()

class_confusions = []
for i in range(len(names_list) - 1):
    for j in range(len(names_list) - 1):
        if i != j and cm[i, j] > 0:
            class_confusions.append({"true": names_list[i], "pred": names_list[j], "count": int(cm[i, j])})
class_conf_df = pd.DataFrame(class_confusions).sort_values("count", ascending=False) if class_confusions else pd.DataFrame()

print("\n=== 배경 관련 혼동 ===")
print(bg_df)
print("\n=== 클래스 간 혼동 ===")
print(class_conf_df)

In [ ]:
box = results.box
per_class = pd.DataFrame({
    "class": [results.names[i] for i in range(len(box.ap50))],
    "precision": box.p,
    "recall": box.r,
    "mAP50": box.ap50,
    "mAP50-95": box.ap,
}).sort_values("recall")

print("\n=== 클래스별 성능 (recall 낮은 순 상위 10개) ===")
print(per_class.head(10))